In [19]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [20]:
PROJECT_ROOT = Path("/content/county-obesity-prediction")
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

modeling_data = pd.read_csv(
    DATA_INTERIM / "modeling_data.csv",
    dtype={"FIPS": "string"}
)

print("Dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())

Dataset shape: (3135, 22)
Unique FIPS: 3135


In [21]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

X = modeling_data[selected_predictors].copy()
y = modeling_data["OBESITY_AdjPrev"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3135, 18)
y shape: (3135,)


In [22]:
RANDOM_STATE = 42

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Outer CV folds:", outer_cv.get_n_splits())

Outer CV folds: 5


In [23]:
def preprocess_outer_fold(X_train_outer, X_val_outer, missing_threshold=20.0):

    missing_summary = pd.DataFrame({
        "missing_count": X_train_outer.isna().sum(),
        "missing_pct": X_train_outer.isna().mean() * 100
    }).round(2)

    excluded_missing = missing_summary[
        missing_summary["missing_pct"] > missing_threshold
    ].index.tolist()

    retained_after_missing = [
        col for col in X_train_outer.columns
        if col not in excluded_missing
    ]

    X_train = X_train_outer[retained_after_missing].copy()
    X_val = X_val_outer[retained_after_missing].copy()

    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_train)

    X_train_imputed = pd.DataFrame(
        imputer.transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_val_imputed = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns,
        index=X_val.index
    )

    corr_matrix = X_train_imputed.corr(method="pearson")

    high_corr_pairs = []

    columns = corr_matrix.columns

    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            r = corr_matrix.iloc[i, j]

            if abs(r) >= 0.80:
                high_corr_pairs.append({
                    "predictor_1": columns[i],
                    "predictor_2": columns[j],
                    "r": r,
                    "abs_r": abs(r)
                })

    high_corr_pairs = pd.DataFrame(high_corr_pairs)

    correlation_excluded = []

    if (
        "POVRATE21" in X_train_imputed.columns
        and "CHILDPOVRATE21" in X_train_imputed.columns
        and abs(
            corr_matrix.loc["POVRATE21", "CHILDPOVRATE21"]
        ) >= 0.80
    ):
        correlation_excluded.append("CHILDPOVRATE21")

    if (
        "PCT_LACCESS_POP19" in X_train_imputed.columns
        and "PCT_LACCESS_LOWI19" in X_train_imputed.columns
        and abs(
            corr_matrix.loc[
                "PCT_LACCESS_POP19",
                "PCT_LACCESS_LOWI19"
            ]
        ) >= 0.80
    ):
        correlation_excluded.append("PCT_LACCESS_POP19")

    retained_after_correlation = [
        col for col in X_train_imputed.columns
        if col not in correlation_excluded
    ]

    X_train_imputed = X_train_imputed[
        retained_after_correlation
    ].copy()

    X_val_imputed = X_val_imputed[
        retained_after_correlation
    ].copy()

    iqr_summary = []

    for column in X_train_imputed.columns:
        values = X_train_imputed[column]

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (values < lower_bound) |
            (values > upper_bound)
        )

        iqr_summary.append({
            "predictor": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_mask.sum()),
            "min": values.min(),
            "max": values.max()
        })

    iqr_summary = pd.DataFrame(iqr_summary)

    implausible_values_removed = 0

    scaler = StandardScaler()
    scaler.fit(X_train_imputed)

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train_imputed),
        columns=X_train_imputed.columns,
        index=X_train_imputed.index
    )

    X_val_scaled = pd.DataFrame(
        scaler.transform(X_val_imputed),
        columns=X_val_imputed.columns,
        index=X_val_imputed.index
    )

    summary = {
        "excluded_missing": excluded_missing,
        "correlation_excluded": correlation_excluded,
        "retained_predictors": X_train_imputed.columns.tolist(),
        "high_corr_pairs": high_corr_pairs,
        "missing_summary": missing_summary,
        "iqr_summary": iqr_summary,
        "iqr_flagged_values": int(
            iqr_summary["outlier_count"].sum()
        ),
        "iqr_predictors_flagged": int(
            (iqr_summary["outlier_count"] > 0).sum()
        ),
        "implausible_values_removed": implausible_values_removed,
        "imputer": imputer,
        "scaler": scaler
    }

    return {
        "X_train_imputed": X_train_imputed,
        "X_val_imputed": X_val_imputed,
        "X_train_scaled": X_train_scaled,
        "X_val_scaled": X_val_scaled,
        "summary": summary
    }

In [24]:
train_idx, val_idx = list(outer_cv.split(X))[0]

X_train_outer = X.iloc[train_idx].copy()
X_val_outer = X.iloc[val_idx].copy()

y_train_outer = y.iloc[train_idx].copy()
y_val_outer = y.iloc[val_idx].copy()

processed = preprocess_outer_fold(
    X_train_outer,
    X_val_outer
)

lr_model = LinearRegression()

lr_model.fit(
    processed["X_train_scaled"],
    y_train_outer
)

y_pred = lr_model.predict(
    processed["X_val_scaled"]
)

mae = mean_absolute_error(y_val_outer, y_pred)
rmse = np.sqrt(mean_squared_error(y_val_outer, y_pred))
r2 = r2_score(y_val_outer, y_pred)

print("Outer Fold 1")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²:", round(r2, 4))
print("Predictors used:", len(processed["summary"]["retained_predictors"]))

Outer Fold 1
MAE: 2.4749
RMSE: 3.1798
R²: 0.5456
Predictors used: 14


In [25]:
lr_fold_results = []
lr_outer_predictions = []

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    X_train_outer = X.iloc[train_idx].copy()
    X_val_outer = X.iloc[val_idx].copy()

    y_train_outer = y.iloc[train_idx].copy()
    y_val_outer = y.iloc[val_idx].copy()

    processed = preprocess_outer_fold(
        X_train_outer,
        X_val_outer
    )

    lr_model = LinearRegression()

    lr_model.fit(
        processed["X_train_scaled"],
        y_train_outer
    )

    y_pred = lr_model.predict(
        processed["X_val_scaled"]
    )

    mae = mean_absolute_error(y_val_outer, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_val_outer, y_pred)
    )
    r2 = r2_score(y_val_outer, y_pred)

    lr_fold_results.append({
        "fold": fold,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    for idx, prediction in zip(val_idx, y_pred):
        lr_outer_predictions.append({
            "FIPS": modeling_data.iloc[idx]["FIPS"],
            "outer_fold": fold,
            "actual": y.iloc[idx],
            "lr_prediction": prediction
        })

lr_results_df = pd.DataFrame(lr_fold_results)

print(lr_results_df.round(4))

   fold     MAE    RMSE      R2
0     1  2.4749  3.1798  0.5456
1     2  2.4559  3.0924  0.5906
2     3  2.4488  3.2093  0.5307
3     4  2.2829  2.9242  0.5859
4     5  2.3902  3.0724  0.4927


In [26]:
lr_summary = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Mean": [
        lr_results_df["MAE"].mean(),
        lr_results_df["RMSE"].mean(),
        lr_results_df["R2"].mean()
    ],
    "SD": [
        lr_results_df["MAE"].std(),
        lr_results_df["RMSE"].std(),
        lr_results_df["R2"].std()
    ]
})

print(lr_summary.round(4))

  Metric    Mean      SD
0    MAE  2.4105  0.0781
1   RMSE  3.0956  0.1117
2     R2  0.5491  0.0407


In [27]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "linear_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

lr_predictions_df = pd.DataFrame(lr_outer_predictions)

lr_results_df.to_csv(
    OUTPUT_DIR / "lr_outer_fold_metrics.csv",
    index=False
)

lr_summary.to_csv(
    OUTPUT_DIR / "lr_performance_summary.csv",
    index=False
)

lr_predictions_df.to_csv(
    OUTPUT_DIR / "lr_outer_predictions.csv",
    index=False
)

print("Saved:")
print("Fold metrics:", lr_results_df.shape)
print("Performance summary:", lr_summary.shape)
print("Outer predictions:", lr_predictions_df.shape)
print("Unique predicted counties:", lr_predictions_df["FIPS"].nunique())

Saved:
Fold metrics: (5, 4)
Performance summary: (3, 3)
Outer predictions: (3135, 4)
Unique predicted counties: 3135
